# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [8]:
!pip -q install transformers sentencepiece torch


In [9]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [10]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1805: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [12]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [13]:
tasks_and_families = {
    "1. Sentiment classification": "Encoder-only (BERT)",
    "2. Machine translation": "Encoder-decoder (T5, BART)",
    "3. Open-ended text generation": "Decoder-only (GPT)",
    "4. Named Entity Recognition": "Encoder-only (BERT)",
    "5. Text summarization": "Encoder-decoder (T5, BART)"
}

for task, family in tasks_and_families.items():
    print(f"{task}: {family}")

1. Sentiment classification: Encoder-only (e.g., BERT)
2. Machine translation: Encoder-decoder (e.g., T5, BART)
3. Open-ended text generation: Decoder-only (e.g., GPT)
4. Named Entity Recognition: Encoder-only (e.g., BERT)
5. Text summarization: Encoder-decoder (e.g., T5, BART)


## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [14]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers."
]

for sentence in sentences:
    result = classifier(sentence)[0]
    print(f"Sentence: '{sentence}'")
    print(f"Label: {result['label']}, Score: {result['score']:.4f}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentence: 'The movie was excellent.'
Label: POSITIVE, Score: 0.9999

Sentence: 'The lecture was difficult to understand.'
Label: NEGATIVE, Score: 0.9997

Sentence: 'I enjoyed learning about transformers.'
Label: POSITIVE, Score: 0.9997



## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

paragraph = """The Amazon rainforest is a moist broadleaf tropical forest in the Amazon biome that covers most of the Amazon basin of South America. This basin encompasses 7,000,000 km2 (2,700,000 sq mi), of which 5,500,000 km2 (2,100,000 sq mi) are covered by the rainforest. This region includes territory belonging to nine nations and 3,344 formally acknowledged indigenous territories. The majority of the forest is contained within Brazil, with 60% of the rainforest, followed by Peru with 13%, Colombia with 10%, and with minor amounts in Venezuela, Ecuador, Bolivia, Guyana, Suriname and French Guiana."""

prompt = f"Summarize the following paragraph in one sentence:\n{paragraph}"

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Original Paragraph:\n", paragraph)
print("\nSummary:\n", summary)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Original Paragraph:
 The Amazon rainforest is a moist broadleaf tropical forest in the Amazon biome that covers most of the Amazon basin of South America. This basin encompasses 7,000,000 km2 (2,700,000 sq mi), of which 5,500,000 km2 (2,100,000 sq mi) are covered by the rainforest. This region includes territory belonging to nine nations and 3,344 formally acknowledged indigenous territories. The majority of the forest is contained within Brazil, with 60% of the rainforest, followed by Peru with 13%, Colombia with 10%, and with minor amounts in Venezuela, Ecuador, Bolivia, Guyana, Suriname and French Guiana.

Summary:
 The Amazon rainforest covers a total area of , and is a major part of the Amazon basin.



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


### Task 5 Reflection

Decoder-only models are highly suitable for chatbots like ChatGPT because they are designed for autoregressive, left-to-right next-token prediction, which matches the natural flow of conversational turn-taking. Since conversations are open-ended, the decoder-only architecture excels at generating fluid, creative, and coherent responses of arbitrary lengths based on the dialogue history. Additionally, processing the entire conversational context as a single continuous sequence is computationally efficient and intuitive within a single model block, eliminating the need for a separate encoder-decoder boundary.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


### Optional Challenge Response

When we remove `set_seed(42)` and enable sampling (e.g., setting `do_sample=True`), we get different outputs because the model no longer selects the single most probable token (greedy decoding). Instead, the model treats the output probabilities for all vocabulary tokens as a probability distribution.

By randomly drawing (sampling) from this probability distribution, the actual token chosen at each step can vary between runs. Because each chosen token changes the context for all subsequent steps, even minor variations in the early tokens will cause the generated text to branch off into completely different sentences.

In [16]:

from transformers import pipeline

generator = pipeline("text-generation", model="distilgpt2")

prompt = "Deep learning is exciting because"

print(f"Prompt: {prompt}\n")
for i in range(3):
    output = generator(prompt, max_new_tokens=20, do_sample=True, temperature=0.7, top_k=50)
    print(f"Generation {i+1}: {output[0]['generated_text']}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: Deep learning is exciting because



[transformers] Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation 1: Deep learning is exciting because it gives us a much more intelligent and open-minded approach to learning and learning. If you are


[transformers] Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation 2: Deep learning is exciting because all the data is being stored in a single file. And this is a very useful technique to understand
Generation 3: Deep learning is exciting because there is so much work to do. We are so excited to be able to put to bed a
